# Open in Colab
<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ai-agents-the-definitive-guide/blob/main/CH11/ch11_memory_footprint_throughput_and_GPU_requirements.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# About the Notebook

### Calculating KV Cache Size per Token for Each Model

At the core of every LLM lies the transformer engine, which consists of two distinct phases: prefill and autoregressive sampling.

- In the prefill phase, the model processes the input prompt tokens in parallel, populating the key-value (KV) cache. The KV cache serves as the model's state, embedded within the attention operation. During this phase, no tokens are generated.
- In the autoregressive sampling phase, the model leverages the current state stored in the KV cache to sample and decode the next token. By reusing the KV cache, the computational overhead of recalculating the cache for every new token is avoided. This approach enables faster sampling, as all previously seen tokens do not need to be processed again through the model.

### Hybrid attention and MoE change the cost math

The models in this notebook (Qwen3.5-4B, Gemma-4-26B-A4B-it, Qwen3.6-35B-A3B) are **hybrid-attention** — only a subset of layers carries a traditional KV cache that grows with context. The remaining layers are either **linear-attention** (fixed-size state, ignored below) or **sliding-window attention** (KV capped at `sliding_window` tokens). Two of them are also **Mixture-of-Experts (MoE)**: total parameters set the memory floor, but only the *active* parameters drive compute and memory bandwidth per token.

The formulas below therefore split two concerns that the classical `4 · n_layers · d_model` rule conflates:

- **Memory footprint** uses `total_params_billion` and only the full-attention (plus sliding) layers contribute growing KV cache.
- **Throughput and latency** use `active_params_billion` — what actually moves through the GPU per token.

### Calculating the Memory Footprint for a Specific Model

Using the hybrid-aware `kv_cache_gib_for_context` helper, it is possible to estimate the memory required to support `n_concurrent_request` sequences at a given context length, accounting for sliding-window caps and MoE total-weight residency.

### Estimating Capacity

State-of-the-art LLM models are memory-bound: performance is limited by memory access bandwidth rather than compute. Batching multiple prompts into the KV cache is the standard technique to improve throughput. Once the maximum number of KV-cache tokens that fit on one or more GPUs is known, the number of concurrent requests follows. Real-world prompts tend to be shorter than `max_context_window`, so actual concurrency is typically higher than the worst-case bound shown.

### Estimating Cost

The final cell translates memory and throughput into **$/1M output tokens** and applies a `CONCURRENT_USAGE_FACTOR` to show the effective cost per *active* user — the number that usually governs unit economics, not the raw GPU hourly rate.

In [ ]:
# =============================================================================
# CONFIG - edit values here; downstream cells read these globals
# =============================================================================
# Bytes per GiB (binary). KV-cache math below is in bytes; outputs are labeled "GiB".
BYTES_IN_GB = 1_073_741_824

# GPU lineup: planning estimates (TFLOPS, bandwidth, and $/hr vary by vendor, region,
# and commitment). $/hr values are indicative on-demand rates as of early 2026 -
# verify against your actual cloud provider pricing before quoting to stakeholders.
GPU_SPECS = [
    {"name": "L4",              "fp16_tflops": 242,  "memory_gb": 24,  "memory_bandwidth_gbps": 300,  "usd_per_hr": 0.70},
    {"name": "L40s",            "fp16_tflops": 362,  "memory_gb": 48,  "memory_bandwidth_gbps": 864,  "usd_per_hr": 1.10},
    {"name": "A100 80 GB PCIe", "fp16_tflops": 312,  "memory_gb": 80,  "memory_bandwidth_gbps": 1935, "usd_per_hr": 1.89},
    {"name": "A100 80 GB SXM",  "fp16_tflops": 312,  "memory_gb": 80,  "memory_bandwidth_gbps": 2039, "usd_per_hr": 2.49},
    {"name": "H100 PCIe",       "fp16_tflops": 756,  "memory_gb": 80,  "memory_bandwidth_gbps": 2000, "usd_per_hr": 3.50},
    {"name": "H100 SXM",        "fp16_tflops": 989,  "memory_gb": 80,  "memory_bandwidth_gbps": 3350, "usd_per_hr": 4.90},
    {"name": "H100 NVL",        "fp16_tflops": 835,  "memory_gb": 94,  "memory_bandwidth_gbps": 3900, "usd_per_hr": 5.90},
    {"name": "H200",            "fp16_tflops": 989,  "memory_gb": 141, "memory_bandwidth_gbps": 4800, "usd_per_hr": 6.50},
    {"name": "B200",            "fp16_tflops": 2250, "memory_gb": 192, "memory_bandwidth_gbps": 8000, "usd_per_hr": 9.99},
]

GPU_MEMORY_GB = {g["name"]: g["memory_gb"] for g in GPU_SPECS}

# Sanity-check defaults.
ESTIMATE_GPU_NAME = "L40s"
DEFAULT_ANALYSIS = dict(num_gpu=4, prompt_sz=4096, response_sz=256, n_concurrent_req=8)

# Fraction of provisioned capacity assumed concurrently active.
# Used in the cost cell to translate raw $/GPU-hr into effective $/active-user.
CONCURRENT_USAGE_FACTOR = 0.10

In [ ]:
# Architecture verified against each model's config.json on Hugging Face.
# Two fields break from the classical schema and drive the 2026 cost story:
#   active_params_billion (!= total for MoE): drives compute/TPOT, not memory.
#   n_full_attention_layers (!= n_layers for hybrids): only these grow KV with context.
#     Other layers are either linear-attention (fixed state, ~0 KV) or
#     sliding-attention (KV capped at sliding_window tokens).
#   Gemma-4 only: full-attn layers use a different KV geometry than sliding layers,
#     so n_global_kv_heads / global_d_head override the defaults for those layers.
MODEL_SPECS = [
    {
        "name": "Qwen3.5-4B",
        "total_params_billion": 4.0,
        "active_params_billion": 4.0,        # dense
        "n_layers": 32,
        "n_full_attention_layers": 8,        # every 4th layer; rest are linear-attn
        "d_model": 2560,
        "n_heads": 16,
        "n_kv_heads": 4,                     # GQA 4:1
        "d_head": 256,
        "sliding_window": None,
        "max_context_window": 262144,
    },
    {
        "name": "ModernBERT-large",          # encoder-only baseline
        "total_params_billion": 0.395,
        "active_params_billion": 0.395,
        "n_layers": 28,
        "n_full_attention_layers": 28,       # full bidirectional attn on every layer
        "d_model": 1024,
        "n_heads": 16,
        "n_kv_heads": 16,                    # MHA
        "d_head": 64,
        "sliding_window": None,
        "max_context_window": 8192,
        "encoder_only": True,
    },
    {
        "name": "Gemma-4-26B-A4B-it",
        "total_params_billion": 26.0,
        "active_params_billion": 4.0,        # 128 experts, top-8
        "n_layers": 30,
        "n_full_attention_layers": 5,        # 1-in-6 pattern; other 25 are sliding_attention
        "d_model": 2816,
        "n_heads": 16,
        "n_kv_heads": 8,                     # sliding layers: GQA 2:1
        "d_head": 256,
        "n_global_kv_heads": 2,              # full-attn layers use a different KV geometry
        "global_d_head": 512,
        "sliding_window": 1024,
        "max_context_window": 262144,
    },
    {
        "name": "Qwen3.6-35B-A3B",
        "total_params_billion": 35.0,
        "active_params_billion": 3.0,        # 256 experts, top-8
        "n_layers": 40,
        "n_full_attention_layers": 10,       # every 4th layer; rest are linear-attn
        "d_model": 2048,
        "n_heads": 16,
        "n_kv_heads": 2,                     # GQA 8:1
        "d_head": 256,
        "sliding_window": None,
        "max_context_window": 262144,
    },
]

In [ ]:
# =============================================================================
# Core formulas (hybrid-attention + MoE aware). Reused by run_llm_analysis and the
# cost cell below.
# =============================================================================

def _full_attn_kv_channels(spec):
    # Gemma-4's full-attn layers use n_global_kv_heads/global_d_head if present;
    # otherwise the full-attn layers use the same geometry as the rest.
    n_kv = spec.get("n_global_kv_heads", spec["n_kv_heads"])
    dh = spec.get("global_d_head", spec["d_head"])
    return n_kv * dh


def kv_growth_per_token_gib(spec):
    # GiB added to KV cache per token once past any sliding window.
    # FP16 (2 bytes) * 2 tensors (K, V) * n_full_attention_layers * KV channels.
    return (2 * 2 * spec["n_full_attention_layers"] * _full_attn_kv_channels(spec)) / BYTES_IN_GB


def kv_cache_gib_for_context(spec, context_tokens):
    # Full-attention layers grow linearly; sliding layers saturate at sliding_window;
    # linear-attention layers carry fixed state (approximated as 0 for sizing).
    full = kv_growth_per_token_gib(spec) * context_tokens
    sw = spec.get("sliding_window")
    if sw:
        n_sliding = spec["n_layers"] - spec["n_full_attention_layers"]
        local_kv_ch = spec["n_kv_heads"] * spec["d_head"]
        sliding_bytes = 2 * 2 * n_sliding * local_kv_ch * min(context_tokens, sw)
        full += sliding_bytes / BYTES_IN_GB
    return full


def memory_footprint_gib(spec, context_tokens, n_concurrent):
    # MoE: weights use total (all experts resident); KV scales with concurrency.
    return spec["total_params_billion"] * 2 + kv_cache_gib_for_context(spec, context_tokens) * n_concurrent


def prefill_ms_per_token(active_params_billion, n_gpu, fp16_tflops):
    # (2 * P_B * 1e9 FLOPs) / (TFLOPS * 1e12) = (2 * P_B / TFLOPS) milliseconds.
    return (2 * active_params_billion / n_gpu) / fp16_tflops


def tpot_ms(active_params_billion, n_gpu, bw_gbps):
    # (2 * P_B GB) / (bw GB/s) = seconds; *1000 -> ms.
    return (2 * active_params_billion / n_gpu) / bw_gbps * 1000


def e2e_latency_s(prefill_ms_val, tpot_ms_val, prompt_sz, response_sz):
    return (prompt_sz * prefill_ms_val + response_sz * tpot_ms_val) / 1000


def max_concurrent_at_context(spec, total_gpu_mem_gib, context_tokens):
    free = total_gpu_mem_gib - spec["total_params_billion"] * 2
    if free <= 0:
        return 0
    per_seq = kv_cache_gib_for_context(spec, context_tokens)
    return int(free // per_seq) if per_seq > 0 else 10**9


def estimate_throughput(num_gpu, prompt_sz, response_sz, n_concurrent_req, model_name="Qwen3.5-4B", gpu_name=None):
    """Quick single-model sanity check using the helpers above."""
    gpu = next(g for g in GPU_SPECS if g["name"] == (gpu_name or ESTIMATE_GPU_NAME))
    spec = next(m for m in MODEL_SPECS if m["name"] == model_name)
    ctx = prompt_sz + response_sz

    total_mem = num_gpu * gpu["memory_gb"]
    mem = memory_footprint_gib(spec, ctx, n_concurrent_req)
    cap = max_concurrent_at_context(spec, total_mem, spec["max_context_window"])
    pft = prefill_ms_per_token(spec["active_params_billion"], num_gpu, gpu["fp16_tflops"])
    tpt = tpot_ms(spec["active_params_billion"], num_gpu, gpu["memory_bandwidth_gbps"])
    lat = e2e_latency_s(pft, tpt, prompt_sz, response_sz)
    tok_s = 1000 / tpt

    print("******************** Estimated LLM Performance ********************")
    print(f"Model: {model_name}  (total={spec['total_params_billion']}B, active={spec['active_params_billion']}B)")
    print(f"GPUs:  {num_gpu}x {gpu['name']}  -  aggregate TFLOPS={num_gpu*gpu['fp16_tflops']:.0f}, "
          f"BW={num_gpu*gpu['memory_bandwidth_gbps']:.0f} GB/s, ${num_gpu*gpu['usd_per_hr']:.2f}/hr")
    print(f"Memory footprint ({n_concurrent_req} concurrent @ {ctx} ctx): {mem:.2f} / {total_mem} GiB")
    print(f"Max concurrent @ max context ({spec['max_context_window']}): {cap}")
    print(f"Prefill / token: {pft:.4f} ms   |   TPOT: {tpt:.2f} ms")
    print(f"E2E latency:     {lat:.2f} s    |   Per-sequence throughput: {tok_s:.2f} tok/s")


estimate_throughput(**DEFAULT_ANALYSIS)

******************** Estimated LLM Performance ********************
Model: Qwen3.5-4B  (total=4.0B, active=4.0B)
GPUs:  4x L40s  -  aggregate TFLOPS=1448, BW=3456 GB/s, $4.40/hr
Memory footprint (8 concurrent @ 4352 ctx): 9.06 / 192 GiB
Max concurrent @ max context (262144): 23
Prefill / token: 0.0055 ms   |   TPOT: 2.31 ms
E2E latency:     0.62 s    |   Per-sequence throughput: 432.00 tok/s


In [ ]:
import csv
from datetime import datetime
from tabulate import tabulate


def run_llm_analysis(num_gpu=1, prompt_sz=4096, response_sz=256, n_concurrent_req=4,
                    gpu_specs=None, model_specs=None):
    """
    Estimate memory footprint, capacity, and latency across every (model, GPU) pair.

    Returns:
        tuple[str, str]: (memory_csv_path, performance_csv_path)
    """
    gpu_specs = gpu_specs if gpu_specs is not None else GPU_SPECS
    model_specs = model_specs if model_specs is not None else MODEL_SPECS
    ctx = prompt_sz + response_sz

    print(f" num_gpu={num_gpu}, prompt_size={prompt_sz} tokens, response_size={response_sz} tokens")
    print(f" n_concurrent_request={n_concurrent_req}")

    # ------------------------------------------------------------------ memory
    print("\n******************** Estimate LLM Memory Footprint ********************")
    memory_table = []
    for m in model_specs:
        kv_seq_gib = kv_cache_gib_for_context(m, ctx)
        footprint = memory_footprint_gib(m, ctx, n_concurrent_req)
        memory_table.append({
            "Model": m["name"],
            "Total / Active (B)": f"{m['total_params_billion']} / {m['active_params_billion']}",
            "Full-attn / layers": f"{m['n_full_attention_layers']} / {m['n_layers']}",
            "Input Size (tokens)": prompt_sz,
            "Output Size (tokens)": response_sz,
            "Concurrent Requests": n_concurrent_req,
            "KV / seq @ ctx": f"{kv_seq_gib*1024:.1f} MiB",
            "Memory Footprint": f"{footprint:.2f} GiB",
        })
    print(tabulate(memory_table, headers="keys", tablefmt="orgtbl"))

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    memory_csv = f"llm_memory_footprint_{timestamp}.csv"
    with open(memory_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=memory_table[0].keys())
        writer.writeheader()
        writer.writerows(memory_table)

    # OOM flags for this (prompt, concurrency) workload
    for m in model_specs:
        footprint = memory_footprint_gib(m, ctx, n_concurrent_req)
        for g in gpu_specs:
            avail = num_gpu * g["memory_gb"]
            if footprint > avail:
                cap = max_concurrent_at_context(m, avail, ctx)
                print(f"  !! OOM  {m['name']} on {num_gpu}x {g['name']} ({avail} GiB): "
                      f"needs {footprint:.1f} GiB. Max concurrent @ this ctx = {cap}")

    # ------------------------------------------------------------ performance
    print("\n******************** Estimate LLM Capacity and Latency ********************")
    perf_table = []
    for m in model_specs:
        for g in gpu_specs:
            avail = num_gpu * g["memory_gb"]
            weight_gib = m["total_params_billion"] * 2
            if weight_gib >= avail:
                perf_table.append({
                    "Model": m["name"], "GPU": g["name"],
                    "Input Size (tokens)": prompt_sz, "Output Size (tokens)": response_sz,
                    "Concurrent Requests": n_concurrent_req,
                    "Max # KV Cache Tokens": "OOM (weights)",
                    "Prefill Time": "OOM", "TPOT (ms)": "OOM", "TTFT": "OOM",
                    "E2E Latency": "OOM", "Output Tokens Throughput": "OOM",
                })
                continue
            free_gib = avail - weight_gib
            growth = kv_growth_per_token_gib(m)
            kv_tokens = int(free_gib / growth) if growth > 0 else 10**9

            pft = prefill_ms_per_token(m["active_params_billion"], num_gpu, g["fp16_tflops"])
            tpt = tpot_ms(m["active_params_billion"], num_gpu, g["memory_bandwidth_gbps"])
            ttft = pft / 1000 + tpt / 1000  # seconds
            e2e = e2e_latency_s(pft, tpt, prompt_sz, response_sz)
            throughput = response_sz / e2e if e2e > 0 else float("inf")

            perf_table.append({
                "Model": m["name"],
                "GPU": g["name"],
                "Input Size (tokens)": prompt_sz,
                "Output Size (tokens)": response_sz,
                "Concurrent Requests": n_concurrent_req,
                "Max # KV Cache Tokens": kv_tokens,
                "Prefill Time": f"{pft:.3f} ms",
                "TPOT (ms)": f"{tpt:.3f} ms",
                "TTFT": f"{ttft:.3f} s",
                "E2E Latency": f"{e2e:.1f} s",
                "Output Tokens Throughput": f"{throughput:.2f} tokens/sec",
            })
    print(tabulate(perf_table, headers="keys", tablefmt="orgtbl"))

    perf_csv = f"llm_performance_{timestamp}.csv"
    with open(perf_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=perf_table[0].keys())
        writer.writeheader()
        writer.writerows(perf_table)

    print(f"\nResults saved to CSV files:\n1. {memory_csv}\n2. {perf_csv}")
    return memory_csv, perf_csv


LAST_MEMORY_CSV, LAST_PERF_CSV = run_llm_analysis(**DEFAULT_ANALYSIS)

 num_gpu=4, prompt_size=4096 tokens, response_size=256 tokens
 n_concurrent_request=8

******************** Estimate LLM Memory Footprint ********************
| Model              | Total / Active (B)   | Full-attn / layers   |   Input Size (tokens) |   Output Size (tokens) |   Concurrent Requests | KV / seq @ ctx   | Memory Footprint   |
|--------------------+----------------------+----------------------+-----------------------+------------------------+-----------------------+------------------+--------------------|
| Qwen3.5-4B         | 4.0 / 4.0            | 8 / 32               |                  4096 |                    256 |                     8 | 136.0 MiB        | 9.06 GiB           |
| ModernBERT-large   | 0.395 / 0.395        | 28 / 28              |                  4096 |                    256 |                     8 | 476.0 MiB        | 4.51 GiB           |
| Gemma-4-26B-A4B-it | 26.0 / 4.0           | 5 / 30               |                  4096 |                    2

In [ ]:
import os
from pathlib import Path

import pandas as pd

def _resolve_analysis_csvs():
    """Prefer paths from the last in-notebook run; else latest CSV pair in cwd."""
    mem, perf = globals().get("LAST_MEMORY_CSV"), globals().get("LAST_PERF_CSV")
    if mem and perf and Path(mem).exists() and Path(perf).exists():
        return str(Path(mem).resolve()), str(Path(perf).resolve())
    roots = [Path.cwd()]
    if Path("/content").exists():
        roots.append(Path("/content"))
    mem_candidates = []
    perf_candidates = []
    for root in roots:
        mem_candidates.extend(root.glob("llm_memory_footprint_*.csv"))
        perf_candidates.extend(root.glob("llm_performance_*.csv"))
    if not mem_candidates or not perf_candidates:
        raise FileNotFoundError(
            "No llm_memory_footprint_*.csv / llm_performance_*.csv found. Run the previous cell first."
        )
    mem = max(mem_candidates, key=lambda p: p.stat().st_mtime)
    perf = max(perf_candidates, key=lambda p: p.stat().st_mtime)
    return str(mem), str(perf)


memory_footprint_file, throughput_file = _resolve_analysis_csvs()
memory_footprint_df = pd.read_csv(memory_footprint_file)
throughput_df = pd.read_csv(throughput_file)
new_memory_footprint_df = memory_footprint_df
new_throughput_df = throughput_df
print(f"Loaded:\n  {throughput_file}\n  {memory_footprint_file}")
throughput_df.head(), memory_footprint_df.head()


Loaded:
  /content/llm_performance_20260418_123727.csv
  /content/llm_memory_footprint_20260418_123727.csv


(        Model              GPU  Input Size (tokens)  Output Size (tokens)  \
 0  Qwen3.5-4B               L4                 4096                   256   
 1  Qwen3.5-4B             L40s                 4096                   256   
 2  Qwen3.5-4B  A100 80 GB PCIe                 4096                   256   
 3  Qwen3.5-4B   A100 80 GB SXM                 4096                   256   
 4  Qwen3.5-4B        H100 PCIe                 4096                   256   
 
    Concurrent Requests  Max # KV Cache Tokens Prefill Time TPOT (ms)     TTFT  \
 0                    8                2883584     0.008 ms  6.667 ms  0.007 s   
 1                    8                6029312     0.006 ms  2.315 ms  0.002 s   
 2                    8               10223616     0.006 ms  1.034 ms  0.001 s   
 3                    8               10223616     0.006 ms  0.981 ms  0.001 s   
 4                    8               10223616     0.003 ms  1.000 ms  0.001 s   
 
   E2E Latency Output Tokens Through

In [ ]:
throughput_df.head()

,Model,GPU,Input Size (tokens),Output Size (tokens),Concurrent Requests,Max # KV Cache Tokens,Prefill Time,TPOT (ms),TTFT,E2E Latency,Output Tokens Throughput
0,Qwen3.5-4B,L4,4096,256,8,2883584,0.008 ms,6.667 ms,0.007 s,1.7 s,147.08 tokens/sec
1,Qwen3.5-4B,L40s,4096,256,8,6029312,0.006 ms,2.315 ms,0.002 s,0.6 s,416.11 tokens/sec
2,Qwen3.5-4B,A100 80 GB PCIe,4096,256,8,10223616,0.006 ms,1.034 ms,0.001 s,0.3 s,880.16 tokens/sec
3,Qwen3.5-4B,A100 80 GB SXM,4096,256,8,10223616,0.006 ms,0.981 ms,0.001 s,0.3 s,922.99 tokens/sec
4,Qwen3.5-4B,H100 PCIe,4096,256,8,10223616,0.003 ms,1.000 ms,0.001 s,0.3 s,959.39 tokens/sec


In [ ]:
# =============================================================================
# COST - translate memory + throughput into $/1M output tokens.
# =============================================================================
# Simplified memory-bandwidth-bound model (matches the rest of the notebook):
#   per-sequence tok/s = 1000 / TPOT_ms          (decode-phase throughput)
#   workload tok/s     = batch * per-sequence    (valid in mem-BW-bound regime)
#   batch              = min(n_concurrent_req, max_concurrent_at_ctx)
#   raw $/1M tok       = (N_gpu * $/hr * 1e6) / (workload tok/s * 3600)
#   effective $/1M     = raw / CONCURRENT_USAGE_FACTOR
#
# Why price at the target workload batch, not max-concurrent: max-concurrent gives
# optimistic aggregate throughput that real servers rarely sustain (compute and
# attention KV-read costs dominate long before you saturate memory capacity). The
# workload-batch number is what you actually pay to serve the target n_concurrent_req.
#
# "Effective" = raw / utilization: you pay the hourly rate continuously, but only
# CONCURRENT_USAGE_FACTOR of provisioned capacity is typically active. This is the
# unit-economics number, not the raw GPU hourly rate.

def serving_cost(num_gpu, spec, gpu, prompt_sz, response_sz, n_concurrent_req):
    ctx = prompt_sz + response_sz
    avail = num_gpu * gpu["memory_gb"]
    weights = spec["total_params_billion"] * 2
    if weights >= avail:
        return {"status": "OOM (weights)"}
    max_batch = max_concurrent_at_context(spec, avail, ctx)
    if max_batch < 1:
        return {"status": "OOM (ctx)"}
    batch = min(n_concurrent_req, max_batch)
    tpt = tpot_ms(spec["active_params_billion"], num_gpu, gpu["memory_bandwidth_gbps"])
    per_seq_tok_s = 1000 / tpt
    workload_tok_s = batch * per_seq_tok_s
    hourly = num_gpu * gpu["usd_per_hr"]
    raw = hourly * 1_000_000 / (workload_tok_s * 3600)
    return {
        "status": "OK" if batch == n_concurrent_req else f"batch_capped@{batch}",
        "batch": batch, "headroom": max_batch,
        "per_seq_tok_s": per_seq_tok_s, "workload_tok_s": workload_tok_s,
        "hourly": hourly, "raw": raw, "effective": raw / CONCURRENT_USAGE_FACTOR,
    }


def cost_table(num_gpu=DEFAULT_ANALYSIS["num_gpu"],
               prompt_sz=DEFAULT_ANALYSIS["prompt_sz"],
               response_sz=DEFAULT_ANALYSIS["response_sz"],
               n_concurrent_req=DEFAULT_ANALYSIS["n_concurrent_req"]):
    eff_label = f"Effective $/1M @ {int(CONCURRENT_USAGE_FACTOR*100)}% active"
    rows = []
    for m in MODEL_SPECS:
        for g in GPU_SPECS:
            r = serving_cost(num_gpu, m, g, prompt_sz, response_sz, n_concurrent_req)
            if r["status"].startswith("OOM"):
                rows.append({
                    "Model": m["name"], "GPUs": f"{num_gpu}x {g['name']}",
                    "Infra $/hr": f"${num_gpu*g['usd_per_hr']:.2f}",
                    "Batch": r["status"], "Headroom (max batch)": "-",
                    "Workload tok/s": "-", "$/1M out tok": "-", eff_label: "-",
                })
            else:
                rows.append({
                    "Model": m["name"],
                    "GPUs": f"{num_gpu}x {g['name']}",
                    "Infra $/hr": f"${r['hourly']:.2f}",
                    "Batch": r["batch"],
                    "Headroom (max batch)": r["headroom"],
                    "Workload tok/s": f"{r['workload_tok_s']:.0f}",
                    "$/1M out tok": f"${r['raw']:.3f}",
                    eff_label: f"${r['effective']:.2f}",
                })
    print(tabulate(rows, headers="keys", tablefmt="orgtbl"))


print(f"Cost estimates for num_gpu={DEFAULT_ANALYSIS['num_gpu']}, "
      f"batch={DEFAULT_ANALYSIS['n_concurrent_req']}, "
      f"ctx={DEFAULT_ANALYSIS['prompt_sz']}+{DEFAULT_ANALYSIS['response_sz']} tokens.")
print(f"Assumption: {int(CONCURRENT_USAGE_FACTOR*100)}% of provisioned capacity is active concurrently.")
print(f"$/hr values are indicative - verify against your actual cloud pricing.\n")
cost_table()

Cost estimates for num_gpu=4, batch=8, ctx=4096+256 tokens.
Assumption: 10% of provisioned capacity is active concurrently.
$/hr values are indicative - verify against your actual cloud pricing.

| Model              | GPUs               | Infra $/hr   |   Batch |   Headroom (max batch) |   Workload tok/s | $/1M out tok   | Effective $/1M @ 10% active   |
|--------------------+--------------------+--------------+---------+------------------------+------------------+----------------+-------------------------------|
| Qwen3.5-4B         | 4x L4              | $2.80        |       8 |                    662 |             1200 | $0.648         | $6.48                         |
| Qwen3.5-4B         | 4x L40s            | $4.40        |       8 |                   1385 |             3456 | $0.354         | $3.54                         |
| Qwen3.5-4B         | 4x A100 80 GB PCIe | $7.56        |       8 |                   2349 |             7740 | $0.271         | $2.71                     